In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
# Dataset cleaning import
df = pd.read_csv('../data/movie_clean.csv')

print("Shape:", df.shape)
df.head(3)

Shape: (4806, 4)


,movie_id,title,tags,tags_clean
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...",century paraplegic marine dispatched moon pand...
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",captain barbossa long believed dead come back ...
2,206647,Spectre,A cryptic message from Bond’s past sends him o...,cryptic message bond past sends trail uncover ...


In [5]:
# Chargement du modèle Sentence-BERT depuis Hugging Face
# 'all-MiniLM-L6-v2' est un modèle léger mais très performant

print("Chargement du modèle...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Modèle chargé !")

Chargement du modèle...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\MOISE\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\MOISE\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modèle chargé !


In [6]:
# Génération d'embeddings : convertit tous les tags_clean en vecteurs numériques
# encode() prend une liste de textes et retourne une matrice numpy
print("Génération d'embeddings...")
embeddings = model.encode(
    df['tags_clean'].tolist(), # convertion de colonne en liste python
    show_progress_bar=True # affiche une barre de progression
)

print(f"\n Shape des embeddings : {embeddings.shape}")

Génération d'embeddings...


Batches:   0%|          | 0/151 [00:00<?, ?it/s]


 Shape des embeddings : (4806, 384)


In [8]:
# On sauvegarde les embeddings
np.save('../data/embeddings.npy', embeddings)
print("embeddings sauvegardés !")

embeddings sauvegardés !


In [12]:
# On Calcule la similarité cosinus entre tous les films
# cosine_similarity compare chaque film avec tous les autres
# Résultat : une matrice (4806 x 4806) où chaque valeur est entre 0 et 1
# 0 = films complètement différents, 1 = films identiques
print("Calcul de la similarité cosinus...")
similarity_matrix = cosine_similarity(embeddings)
print(f"Shape de la matrice : {similarity_matrix.shape}")
print(f"Similarité Avatar avec lui-même : {similarity_matrix[0][0]:.2f}")

Calcul de la similarité cosinus...
Shape de la matrice : (4806, 4806)
Similarité Avatar avec lui-même : 1.00


In [11]:
similarity_matrix

array([[1.        , 0.48470908, 0.33866823, ..., 0.35438672, 0.3097577 ,
        0.126721  ],
       [0.48470908, 1.        , 0.38191342, ..., 0.4589694 , 0.40441665,
        0.35082093],
       [0.33866823, 0.38191342, 0.99999976, ..., 0.44320166, 0.33873254,
        0.11992767],
       ...,
       [0.35438672, 0.4589694 , 0.44320166, ..., 1.        , 0.3437064 ,
        0.14585802],
       [0.3097577 , 0.40441665, 0.33873254, ..., 0.3437064 , 1.        ,
        0.22814722],
       [0.126721  , 0.35082093, 0.11992767, ..., 0.14585802, 0.22814722,
        1.        ]], dtype=float32)

In [19]:
# Recommandation
# On crée une fonction de recommandation
def recommend(title, n=5):
    """
    Recommande n films similaires à un film donné.
    title : nom du film (doit exister dans le dataset)
    n     : nombre de recommandations (5 par défaut)
    """
    if title not in df['title'].values:
        return f"Film '{title}' non trouvé dans le dataset"
    
    # on récupère l'index du film dans le dataframe
    idx = df[df['title'] == title].index[0]

    # On récupère les scores de similarité de ce film avec tous les autres
    # similarity_matrix[idx] = toute la ligne du film dans la matrice
    # chaque tuple = (index_film, score_similarité)
    scores = list(enumerate(similarity_matrix[idx]))

    # On trie les films par score décroissant (du plus similaire au moins)
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    # On ignore le premier résultat car c'est le film lui-même (score = 1.0)
    # On garde les n films suivants
    top_n = scores[1:n+1]

    # On récupère les titres des films recommandés
    recommended = [df.iloc[i[0]]['title'] for i in top_n]

    return recommended

In [20]:
# test avec avatar
print("Films similaires à Avatar :")
print(recommend("Avatar"))

Films similaires à Avatar :
['Serenity', 'Star Trek Into Darkness', 'Aliens', 'Lost in Space', 'After Earth']


In [21]:
# Test avec différents genres pour vérifier la cohérence
print("Films similaires à The Dark Knight :")
print(recommend("The Dark Knight"))

print("\nFilms similaires à The Godfather :")
print(recommend("The Godfather"))

print("\nFilms similaires à Titanic :")
print(recommend("Titanic"))

Films similaires à The Dark Knight :
['The Dark Knight Rises', 'Batman', 'Batman', 'Batman Begins', 'Batman & Robin']

Films similaires à The Godfather :
['The Godfather: Part II', 'The Godfather: Part III', 'The Last Godfather', "Things to Do in Denver When You're Dead", 'Loose Cannons']

Films similaires à Titanic :
['The Ballad of Jack and Rose', "Fool's Gold", "Pirates of the Caribbean: At World's End", 'The Chambermaid on the Titanic', 'Poseidon']


In [22]:
# Sauvegarde de la matrice en cache
np.save('../data/similarity_matrix.npy', similarity_matrix)
print("Matrice de similarité sauvegardée !")

Matrice de similarité sauvegardée !
